# S8 - Procesamiento en streaming con Spark: ventanas, watermarking y semántica de entrega

**Proyecto Sello — Equipo LLSW3 — Simulación de tráfico de red**

**Actividad:** construir un pipeline de Spark Structured Streaming sobre `red-telemetria` (el topic de Kafka que S7 ya deja publicando telemetría de tráfico de red), aplicando agregaciones por ventana de tiempo, watermarking para datos tardíos, checkpointing para tolerancia a fallos, deduplicación acotada, y comparando semántica de entrega y throughput/latencia bajo distintos parámetros de disparo.

**Arquitectura del sistema:**
```
NIC física (Ethernet) ─┐
NIC virtuales ─────────┤  agente/agente_red.py   (Windows, fuera de Docker)
                       ▼  MQTT  lambda26/red/llsw3/telemetria
               mosquitto (contenedor, puerto 41883 del host)
                       ▼
               app/bridge_mqtt_kafka.py  → Kafka: red-telemetria (3 particiones, key = sensorId)
                       ▼
               Spark Structured Streaming ← (este notebook)
               ventanas + watermark + checkpoint + deduplicación
```

**Contrato del evento `red.lectura`:**
```json
{
  "tipoEvento": "red.lectura",
  "sensorId":   "Ethernet",
  "bytesSentPerSec": 1234.5,
  "bytesRecvPerSec": 5678.9,
  "pktsSentPerSec":  10.0,
  "pktsRecvPerSec":  25.0,
  "errin": 0, "errout": 0, "dropin": 0, "dropout": 0,
  "velocidadEnlaceMbps": 1000.0,
  "timestamp": 1790512778045
}
```

## Objetivo

1. Verificar que Kafka y el bridge/agente de S7 siguen funcionando, y unir `pyspark` a la red de Kafka.
2. Crear la `SparkSession` con el conector de Kafka.
3. Leer `red-telemetria` en modo streaming y parsear el JSON con un esquema explícito adaptado al tráfico de red.
4. Ver los eventos llegar en vivo, por consola, y persistir esa misma lectura en Parquet.
5. Agregar tráfico por ventana de tiempo (promedio de bytes/seg y paquetes/seg), con y sin watermarking, y comparar.
6. Agregar por ventana deslizante e inspeccionar el contenido real de un checkpoint.
7. Provocar un evento tardío descartado por el watermark y un duplicado filtrado por `dropDuplicates`.
8. Comparar el costo de dos intervalos de disparo distintos.

## 3.1 Reanudar el entorno del Proyecto Sello y verificar el punto de partida

**Producto del paso:** Kafka, Kafka UI, mosquitto y el bridge MQTT→Kafka (`uso-red`) corriendo, con eventos de tráfico de red llegando a `red-telemetria`; y `pyspark` unido a la red de Kafka.

Este notebook **no genera sus propios datos**: consume el mismo topic que S7 ya construyó. Antes de continuar, en una terminal (fuera de este notebook):

```powershell
# Terminal 1: levantar Kafka
cd tareaproyectosello
docker compose -f kafka/compose.yml up -d

# Crear el topic si no existe (3 particiones, key = sensorId)
docker compose -f kafka/compose.yml exec kafka /opt/kafka/bin/kafka-topics.sh `
  --create --topic red-telemetria `
  --partitions 3 --replication-factor 1 `
  --bootstrap-server kafka:9092 `
  --if-not-exists

# Terminal 2: levantar uso-red (mosquitto + bridge)
docker compose -f uso-red/compose.yml up -d --build

# Terminal 3: levantar pyspark conectado a la red de Kafka
docker compose -f pyspark/compose.yml up -d --build
```

Verificar que el topic tiene mensajes (necesita que el agente y el bridge estén corriendo):

```powershell
# Arrancar el consumer para ver eventos en vivo (Ctrl+C cuando ya viste suficiente)
docker compose -f uso-red/compose.yml exec uso-red python /app/consumer_red.py

# Arrancar el bridge MQTT->Kafka (dejar corriendo)
docker compose -f uso-red/compose.yml exec uso-red python /app/bridge_mqtt_kafka.py

# Arrancar el agente en Windows (dejar corriendo)
python -u uso-red/agente/agente_red.py
```

**El agente publica una lectura por segundo de cada NIC activa** (Ethernet, Wi-Fi, etc.). Mientras esté corriendo, el topic `red-telemetria` crece continuamente — exactamente lo que las celdas de streaming de este notebook necesitan.

**Error frecuente**: si las celdas de streaming no muestran filas nuevas, confirma que el agente (`agente_red.py`) sigue corriendo en Windows y que el bridge sigue imprimiendo `"status": "forwarded"`. Sin esos dos procesos, no hay datos nuevos en `red-telemetria`.

## 3.2 Crear la `SparkSession` con el conector de Kafka

In [24]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s08-streaming-red-trafico")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")  # igual que las 3 particiones del topic
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

`spark.jars.packages` agrega el conector `spark-sql-kafka-0-10`, que Spark no trae por defecto. La primera vez que se ejecuta esta celda, Spark descarga el conector y sus dependencias desde Maven Central — puede tardar. `spark.sql.shuffle.partitions` se baja a 3, igual que las 3 particiones del topic `red-telemetria`.

Verificar que el topic tiene mensajes, con una lectura **batch** (no streaming):

In [25]:
lote = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "red-telemetria")
    .option("startingOffsets", "earliest")
    .load()
)
print("mensajes en el topic red-telemetria:", lote.count())
lote.selectExpr(
    "CAST(key AS STRING) AS sensorId",
    "CAST(value AS STRING) AS value",
    "partition",
    "offset"
).orderBy("offset", ascending=False).show(3, truncate=False)

mensajes en el topic red-telemetria: 18175
+----------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|sensorId                          |value                                                                                                                                                                                                                                                                                                                 |partition|offset|
+----------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Resultado esperado: un número mayor que cero, y en las últimas filas, JSON con `sensorId`, `bytesSentPerSec`, `bytesRecvPerSec`, etc. (el contrato `red.lectura` de S7). Una lectura **batch** de Kafka lee lo que hay hoy y termina; el resto del notebook usa lectura **streaming**, que no termina — sigue escuchando.

## 3.3 Leer el topic en modo streaming

**Producto del paso:** un DataFrame que **no se puede materializar con `.show()` ni `.count()`** — la primera diferencia real entre batch y streaming.

In [26]:
crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "red-telemetria")
    .option("startingOffsets", "latest")
    .load()
)
print("es un stream:", crudo.isStreaming)
crudo.printSchema()

es un stream: True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



`readStream` en vez de `read` es la única diferencia de sintaxis, y cambia todo lo que viene después: `crudo` es un **plan**, no una tabla en memoria. `crudo.show()` fallaría con `AnalysisException`. Cada mensaje de Kafka llega con las mismas columnas: `key` (= sensorId, en binario), `value` (el JSON del evento, en binario), `topic`, `partition`, `offset`, `timestamp`. El contenido real del evento está **adentro** de `value`, sin parsear — eso es 3.4.

`startingOffsets: "latest"` es a propósito: en un stream indefinido, no tiene sentido reprocesar todo el historial cada vez que se reinicia una consulta de prueba.

## 3.4 Parsear el JSON con un esquema explícito adaptado a tráfico de red

**Producto del paso:** las columnas reales del evento de tráfico de red (`sensorId`, `bytesSentPerSec`, `bytesRecvPerSec`...), ya tipadas, listas para agregar.

In [27]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, IntegerType

# Esquema completo del evento red.lectura (contrato de S7)
esquema_red = (
    StructType()
    .add("tipoEvento",          StringType())
    .add("sensorId",            StringType())   # nombre de la NIC: "Ethernet", "Wi-Fi", etc.
    .add("bytesSentPerSec",     DoubleType())   # bytes enviados por segundo
    .add("bytesRecvPerSec",     DoubleType())   # bytes recibidos por segundo
    .add("pktsSentPerSec",      DoubleType())   # paquetes enviados por segundo
    .add("pktsRecvPerSec",      DoubleType())   # paquetes recibidos por segundo
    .add("errin",               DoubleType())   # errores de entrada (acumulados)
    .add("errout",              DoubleType())   # errores de salida (acumulados)
    .add("dropin",              DoubleType())   # descartes de entrada (acumulados)
    .add("dropout",             DoubleType())   # descartes de salida (acumulados)
    .add("velocidadEnlaceMbps", DoubleType())   # capacidad de la NIC en Mbps
    .add("timestamp",           LongType())     # epoch en milisegundos
)

eventos = (
    crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema_red).alias("e"))
    .select("e.*")
    # Convertir timestamp (ms epoch Long) → tipo Timestamp real para las ventanas
    .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
    # Calcular uso del enlace como porcentaje de capacidad
    .withColumn(
        "uso_enlace_pct",
        ((col("bytesSentPerSec") + col("bytesRecvPerSec")) /
         (col("velocidadEnlaceMbps") * 125000.0) * 100.0)
    )
)
eventos.printSchema()

root
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- bytesSentPerSec: double (nullable = true)
 |-- bytesRecvPerSec: double (nullable = true)
 |-- pktsSentPerSec: double (nullable = true)
 |-- pktsRecvPerSec: double (nullable = true)
 |-- errin: double (nullable = true)
 |-- errout: double (nullable = true)
 |-- dropin: double (nullable = true)
 |-- dropout: double (nullable = true)
 |-- velocidadEnlaceMbps: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- uso_enlace_pct: double (nullable = true)



`ts` convierte el `timestamp` (milisegundos epoch, un `Long`) a un tipo `timestamp` real: **es la columna que 3.5 en adelante usa como reloj del evento** — el momento en que el agente leyó la NIC, no el momento en que Spark lo procesa.

`uso_enlace_pct` es una métrica derivada: qué porcentaje de la capacidad de la NIC se está usando. `velocidadEnlaceMbps × 125000` convierte Mbps → bytes/segundo (1 Mbps = 125 000 B/s).

## 3.5 Primera consulta: ver los eventos de tráfico llegar

**Producto del paso:** la primera consulta de streaming corriendo, con salida por consola — eventos de tráfico de red en vivo.

In [28]:
# Momento 1: arrancar la consulta
consulta = (
    eventos.select(
        "sensorId", "bytesSentPerSec", "bytesRecvPerSec",
        "pktsSentPerSec", "pktsRecvPerSec",
        "uso_enlace_pct", "ts"
    )
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

`writeStream` inicia una **consulta continua**: `.start()` la lanza en segundo plano y devuelve de inmediato un objeto `StreamingQuery`; el proceso sigue corriendo aunque la celda ya haya terminado. `outputMode("append")` muestra solo filas nuevas. `trigger(processingTime="5 seconds")` fija cada cuánto se arma un micro-lote.

Déjala corriendo — **no la detengas todavía**. En 3.6 vas a arrancar una segunda consulta que persiste en Parquet al mismo tiempo.

## 3.6 Escribir la salida a Parquet, con checkpoint

**Producto del paso:** los eventos de tráfico de red aterrizando en disco, en Parquet, listos para análisis posteriores (S9-S10).

In [29]:
# Momento 1: arrancar el stream a Parquet (la consulta de 3.5 sigue corriendo en paralelo)
ARTIFACTS      = "./artifacts/red_parquet"
CHECKPOINT_PARQUET = "./artifacts/chk_red_parquet"

consulta2 = (
    eventos.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", ARTIFACTS)
    .option("checkpointLocation", CHECKPOINT_PARQUET)
    .trigger(processingTime="5 seconds")
    .start()
)

In [30]:
# Momento 2 (opcional): leer lo guardado (se puede repetir, el stream sigue corriendo)
import time
time.sleep(10)  # esperar al menos un micro-lote
guardado = spark.read.parquet(ARTIFACTS)
total = guardado.count()
print("filas guardadas en Parquet:", total)
guardado.select("sensorId", "bytesSentPerSec", "bytesRecvPerSec", "uso_enlace_pct", "ts").show(10, truncate=False)

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+---------------+---------------+--------------+--------------+--------------+---+
|sensorId|bytesSentPerSec|bytesRecvPerSec|pktsSentPerSec|pktsRecvPerSec|uso_enlace_pct|ts |
+--------+---------------+---------------+--------------+--------------+--------------+---+
+--------+---------------+---------------+--------------+--------------+--------------+---+

-------------------------------------------
Batch: 1
-------------------------------------------
+----------------------------------+------------------+------------------+-----------------+------------------+------------------+-----------------------+
|sensorId                          |bytesSentPerSec   |bytesRecvPerSec   |pktsSentPerSec   |pktsRecvPerSec    |uso_enlace_pct    |ts                     |
+----------------------------------+------------------+------------------+-----------------+------------------+----------------

-------------------------------------------
Batch: 2
-------------------------------------------
+----------------------------------+-----------------+------------------+------------------+-----------------+------------------+-----------------------+
|sensorId                          |bytesSentPerSec  |bytesRecvPerSec   |pktsSentPerSec    |pktsRecvPerSec   |uso_enlace_pct    |ts                     |
+----------------------------------+-----------------+------------------+------------------+-----------------+------------------+-----------------------+
|Ethernet 2                        |0.0              |0.0               |0.0               |0.0              |0.0               |2026-10-08 18:18:45.626|
|Ethernet 2                        |0.0              |0.0               |0.0               |0.0              |0.0               |2026-10-08 18:18:45.626|
|Ethernet 2                        |0.0              |0.0               |0.0               |0.0              |0.0               |2026

In [31]:
# Momento 3: detener ambas consultas (la de consola de 3.5 y la de Parquet de 3.6)
import time
time.sleep(20)
consulta.stop()
consulta2.stop()

-------------------------------------------
Batch: 3
-------------------------------------------
+----------+---------------+---------------+--------------+--------------+--------------+-----------------------+
|sensorId  |bytesSentPerSec|bytesRecvPerSec|pktsSentPerSec|pktsRecvPerSec|uso_enlace_pct|ts                     |
+----------+---------------+---------------+--------------+--------------+--------------+-----------------------+
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:18:48.92 |
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:18:48.92 |
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:18:49.705|
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:18:49.705|
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:18

-------------------------------------------
Batch: 6
-------------------------------------------
+----------+---------------+---------------+--------------+--------------+--------------+-----------------------+
|sensorId  |bytesSentPerSec|bytesRecvPerSec|pktsSentPerSec|pktsRecvPerSec|uso_enlace_pct|ts                     |
+----------+---------------+---------------+--------------+--------------+--------------+-----------------------+
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:19:04.986|
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:19:04.986|
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:19:05.135|
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:19:05.135|
|Ethernet 2|0.0            |0.0            |0.0           |0.0           |0.0           |2026-10-08 18:19

Un *sink* de archivo (Parquet) solo acepta `outputMode("append")` — no puede "reescribir" una fila ya guardada en disco.

**Error frecuente**: si lees muy rápido, antes del primer micro-lote, el resultado es cero filas. Espera unos segundos y vuelve a correr la celda de lectura; si sigue en 0, confirma que el agente y el bridge siguen corriendo.

## 3.7 Agregar tráfico por ventana de tiempo, sin límite (el problema que resuelve 3.8)

**Producto del paso:** evidencia de que, sin watermark, el estado de una agregación por ventana **crece para siempre**.

La métrica aquí es el **promedio de bytes/seg enviados y recibidos**, y el **promedio de uso del enlace**, agrupados por NIC (`sensorId`) y ventana de 10 segundos.

In [32]:
# Momento 1: arrancar la agregación sin límite
from pyspark.sql.functions import window, avg, count, max as spark_max

agregado_sin_limite = (
    eventos.groupBy(
        window(col("ts"), "10 seconds"),
        col("sensorId")
    )
    .agg(
        avg("bytesSentPerSec").alias("avg_sent_Bps"),
        avg("bytesRecvPerSec").alias("avg_recv_Bps"),
        avg("uso_enlace_pct").alias("avg_uso_pct"),
        count("*").alias("n")
    )
)

consulta = (
    agregado_sin_limite.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

Déjala correr al menos 20-25 segundos (varios micro-lotes) para que se note la tabla creciendo, y después detenla:

In [33]:
# Momento 2: detener la consulta
import time
time.sleep(20)
consulta.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+------------+------------+-----------+---+
|window|sensorId|avg_sent_Bps|avg_recv_Bps|avg_uso_pct|n  |
+------+--------+------------+------------+-----------+---+
+------+--------+------------+------------+-----------+---+

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-08 18:19:10, 2026-10-08 18:19:20}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0             

`window(col("ts"), "10 seconds")` agrupa los eventos en bloques de 10 segundos de tiempo de **evento** (`ts`), no de tiempo de procesamiento. `outputMode("complete")` es el único modo válido para una agregación **sin** watermark: en cada micro-lote, Spark reimprime **todas** las ventanas vistas hasta ahora.

Observa que, de un micro-lote al siguiente, la tabla completa **crece**: las ventanas de hace 20 segundos siguen apareciendo, iguales, junto a las nuevas. En un sistema real monitoreando tráfico de red por días, esto agota la memoria. El watermarking (3.8) es la respuesta.

## 3.8 Watermarking: acotar cuánto se espera por datos tardíos

**Producto del paso:** la misma agregación de tráfico de red, pero con las ventanas viejas **cerradas y liberadas** de la memoria.

In [34]:
# Momento 1: arrancar la agregación con watermark
agregado_con_watermark = (
    eventos
    .withWatermark("ts", "10 seconds")
    .groupBy(
        window(col("ts"), "10 seconds"),
        col("sensorId")
    )
    .agg(
        avg("bytesSentPerSec").alias("avg_sent_Bps"),
        avg("bytesRecvPerSec").alias("avg_recv_Bps"),
        avg("uso_enlace_pct").alias("avg_uso_pct"),
        count("*").alias("n")
    )
)

consulta = (
    agregado_con_watermark.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-watermark-red")
    .trigger(processingTime="5 seconds")
    .start()
)

Déjala correr al menos 20-25 segundos y detenla:

In [35]:
# Momento 2: detener la consulta
import time
time.sleep(20)
consulta.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+------------+------------+-----------+---+
|window|sensorId|avg_sent_Bps|avg_recv_Bps|avg_uso_pct|n  |
+------+--------+------------+------------+-----------+---+
+------+--------+------------+------------+-----------+---+

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------------------------------+-----------------+------------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps     |avg_recv_Bps      |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+-----------------+------------------+--------------------+---+
|{2026-10-08 18:19:30, 2026-10-08 18:19:40}|Ethernet                          |144.4793849281017|171.59647688940612|2.528606894540063E-4|16

`withWatermark("ts", "10 seconds")` le dice a Spark: "una vez que veas un evento con `ts` = X, ya no esperes datos con `ts` anterior a X menos 10 segundos — cierra esas ventanas y olvídalas". Con el watermark, `outputMode("update")` ya es válido: en cada micro-lote solo se imprimen las ventanas que **cambiaron**.

Compara la salida con la de 3.7: aquí la tabla **no** repite ventanas viejas en cada lote — cada micro-lote solo trae lo nuevo o lo que cambió. Las ventanas de tráfico cerradas dejan de ocupar memoria.

## 3.9 Ventana deslizante: monitoreo de tráfico que se actualiza más seguido

**Producto del paso:** ventanas que se **solapan**, para suavizar la métrica de tráfico sin esperar a que cada bloque de 15 segundos cierre — útil para detectar picos de tráfico antes de que la ventana entera termine.

In [36]:
# Momento 1: arrancar la ventana deslizante
# Ventana de 15s que se actualiza cada 5s: captura picos de tráfico más rápido
agregado_deslizante = (
    eventos
    .withWatermark("ts", "15 seconds")
    .groupBy(
        window(col("ts"), "15 seconds", "5 seconds"),
        col("sensorId")
    )
    .agg(
        avg("bytesSentPerSec").alias("avg_sent_Bps"),
        avg("bytesRecvPerSec").alias("avg_recv_Bps"),
        avg("uso_enlace_pct").alias("avg_uso_pct"),
        count("*").alias("n")
    )
)

consulta = (
    agregado_deslizante.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-deslizante-red")
    .trigger(processingTime="5 seconds")
    .start()
)

Déjala correr al menos 20-25 segundos y detenla:

In [37]:
# Momento 2: detener la consulta
import time
time.sleep(20)
consulta.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+------------+------------+-----------+---+
|window|sensorId|avg_sent_Bps|avg_recv_Bps|avg_uso_pct|n  |
+------+--------+------------+------------+-----------+---+
+------+--------+------------+------------+-----------+---+

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+----------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps    |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+------------------+----------------+--------------------+---+
|{2026-10-08 18:19:45, 2026-10-08 18:20:00}|Ethernet                          |54.254107645487196|65.0919269455436|9.547682767282464E-5|8  |
|

`window(col("ts"), "15 seconds", "5 seconds")` tiene un tercer argumento (el **paso** de 5 segundos, menor que el **ancho** de 15 segundos). El resultado es una ventana **deslizante**: cada evento de tráfico cae en **varias** ventanas de 15 segundos a la vez.

Observa en la salida que, para un mismo `sensorId` (p.ej. `Ethernet`), aparecen varias filas con rangos de `window` que se superponen — el mismo evento de tráfico cuenta para varias de ellas. Esto sirve para detectar picos de tráfico antes de que la ventana completa cierre, a costa de guardar más estado.

## 3.10 Checkpointing: qué guarda, y qué pasa al reiniciar

**Producto del paso:** evidencia de que el *checkpoint* no es una carpeta vacía — ahí vive el progreso real de la consulta de tráfico de red.

In [38]:
import os

chk_path = "/tmp/s08-chk-watermark-red"
print("Contenido del checkpoint de la consulta 3.8:")
for carpeta in sorted(os.listdir(chk_path)):
    print(" ", carpeta)

Contenido del checkpoint de la consulta 3.8:
  .metadata.crc
  commits
  metadata
  offsets
  sources
  state


In [39]:
# Leer el último offset guardado — qué tan lejos llegó Spark en el topic red-telemetria
archivos_offsets = sorted(os.listdir(f"{chk_path}/offsets"))
ultimo_offset = archivos_offsets[-1]
print(f"Último offset guardado ({ultimo_offset}):")
with open(f"{chk_path}/offsets/{ultimo_offset}") as f:
    print(f.read())

Último offset guardado (4):
v1
{"batchWatermarkMs":1791483579733,"batchTimestampMs":1791483595004,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.stateStore.rocksdb.mergeOperatorVersion":"2","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.queryEvolution.enableSourceEvolution":"false","spark.sql.streaming.stateStore.encodingFormat":"unsaferow","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.stateStore.rowChecksum.enabled":"false","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.function.protobufExtensions.enabled":"false","spark.sql.shuffle.partitions":"3","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.op

`offsets/` guarda, por cada micro-lote, hasta qué *offset* de cada partición de Kafka se leyó — con el topic de 3 particiones (`red-telemetria`), vas a ver tres claves (0, 1, 2). `commits/` confirma cuáles micro-lotes terminaron. `state/` guarda el contenido de cada ventana de tráfico abierta.

La consecuencia práctica: si la consulta de monitoreo de tráfico se cae y se reinicia apuntando al mismo `checkpointLocation`, Spark retoma exactamente donde se quedó — no reprocesa todo `red-telemetria` desde el principio, y no pierde el estado de las ventanas que ya tenía abiertas.

## 3.11 Un evento de tráfico tardío, y el watermark descartándolo

**Producto del paso:** un evento con marca de tiempo vieja (45 segundos en el pasado), publicado **después** de que el watermark ya avanzó más allá de su ventana — y la prueba de que Spark lo descarta sin avisar.

**Advertencia sobre el "arranque en frío":** si publicas el evento tardío como el **primer** mensaje que la consulta ve, no se descarta — el watermark todavía no tiene ningún evento "reciente" contra el cual compararlo. Por eso, primero se deja correr la consulta con datos en vivo un rato, y **recién después** se publica el evento tardío.

In [40]:
# Momento 1: arrancar el hilo y la consulta
import threading
import time
import json
from pyspark.sql.functions import to_json, struct

def publicar_evento_tardio_red(segundos_de_espera, segundos_en_el_pasado):
    time.sleep(segundos_de_espera)
    ahora_ms = int(time.time() * 1000)
    # Evento de tráfico de red con timestamp 45 segundos en el pasado
    evento_tardio = [(
        "red.lectura",
        "nic-tardio",       # NIC ficticia para identificarla fácilmente
        999999.9,           # bytesSentPerSec exagerado — fácil de ver si aparece
        999999.9,           # bytesRecvPerSec
        9999.9,             # pktsSentPerSec
        9999.9,             # pktsRecvPerSec
        0.0, 0.0, 0.0, 0.0, # errin, errout, dropin, dropout
        1000.0,             # velocidadEnlaceMbps
        ahora_ms - segundos_en_el_pasado * 1000   # timestamp 45s en el pasado
    )]
    columnas = [
        "tipoEvento", "sensorId", "bytesSentPerSec", "bytesRecvPerSec",
        "pktsSentPerSec", "pktsRecvPerSec",
        "errin", "errout", "dropin", "dropout",
        "velocidadEnlaceMbps", "timestamp"
    ]
    df_tardio = spark.createDataFrame(evento_tardio, columnas)
    salida = df_tardio.select(
        df_tardio.sensorId.cast("string").alias("key"),
        to_json(struct(*columnas)).alias("value"),
    )
    (salida.write.format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("topic", "red-telemetria")
        .save())
    print(f">>> publicado nic-tardio, {segundos_en_el_pasado}s en el pasado (bytesSent=999999.9)")

hilo = threading.Thread(target=publicar_evento_tardio_red, args=(15, 45), daemon=True)
hilo.start()

# Consulta con watermark estricto de 5 segundos para que el tardío quede fuera
agregado_tardio = (
    eventos
    .withWatermark("ts", "5 seconds")
    .groupBy(
        window(col("ts"), "10 seconds"),
        col("sensorId")
    )
    .agg(count("*").alias("n"))
)

consulta = (
    agregado_tardio.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-tardio-red")
    .trigger(processingTime="4 seconds")
    .start()
)

El hilo publica el evento tardío 15 segundos después de arrancar la consulta. Espera **al menos 30-40 segundos** antes de detenerla — necesitas ver varios micro-lotes con tráfico real antes del evento tardío, y varios más después, para comprobar que `nic-tardio` nunca aparece en la salida.

In [41]:
# Momento 2: detener la consulta y revisar
import time
time.sleep(20)
consulta.stop()
print("Buscando 'nic-tardio' en la salida de arriba: si no aparece ninguna fila, el watermark lo descartó.")

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+---+
|window|sensorId|n  |
+------+--------+---+
+------+--------+---+

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------------------------------+---+
|window                                    |sensorId                          |n  |
+------------------------------------------+----------------------------------+---+
|{2026-10-08 18:20:10, 2026-10-08 18:20:20}|Ethernet 2                        |4  |
|{2026-10-08 18:20:20, 2026-10-08 18:20:30}|Ethernet 2                        |4  |
|{2026-10-08 18:20:10, 2026-10-08 18:20:20}|vEthernet (WSL (Hyper-V firewall))|4  |
|{2026-10-08 18:20:20, 2026-10-08 18:20:30}|Ethernet                          |4  |
|{2026-10-08 18:20:10, 2026-10-08 18:20:20}|Ethernet                          |4  |
|{2026-10-08 18:20:20, 2026-10-08 18:20:30}|v

26/10/08 18:20:40 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 6, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
26/10/08 18:20:40 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 6, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.


El hilo publica un evento `nic-tardio` cuyo `timestamp` corresponde a **45 segundos antes** del momento de publicación, pero recién 15 segundos después de arrancar la consulta — tiempo suficiente para que el watermark ya haya avanzado con los eventos en vivo del agente de red.

Con un watermark de 5 segundos, cualquier ventana que termine más de 5 segundos antes del evento más reciente visto queda cerrada, y un evento que llega después para esa ventana se descarta **en silencio**: no hay excepción, no hay log de error, la fila simplemente nunca aparece en la salida.

## 3.12 Deduplicación acotada por watermark

**Producto del paso:** el mismo evento de tráfico de red, publicado dos veces por error, contado **una sola vez**.

Esto modela el caso real de Kafka: si el bridge MQTT→Kafka reintenta un envío (por un timeout de red), el mismo evento de tráfico puede llegar duplicado al topic `red-telemetria`.

In [42]:
# Momento 1: arrancar la consulta
# dropDuplicates por sensorId + timestamp: la misma lectura de la misma NIC en el mismo instante
deduplicado = (
    eventos
    .withWatermark("ts", "30 seconds")
    .dropDuplicates(["sensorId", "timestamp"])
)

consulta = (
    deduplicado.select("sensorId", "bytesSentPerSec", "uso_enlace_pct", "ts")
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/s08-chk-dedup-red")
    .trigger(processingTime="3 seconds")
    .start()
)

Con la consulta corriendo, espera unos segundos a que arranque y publica el mismo evento de tráfico dos veces seguidas:

In [43]:
# Momento 2: publicar el evento duplicado (simula reintento del bridge MQTT→Kafka)
time.sleep(6)
columnas = [
    "tipoEvento", "sensorId", "bytesSentPerSec", "bytesRecvPerSec",
    "pktsSentPerSec", "pktsRecvPerSec",
    "errin", "errout", "dropin", "dropout",
    "velocidadEnlaceMbps", "timestamp"
]
# Evento duplicado: mismo sensorId, mismo timestamp (simula reintento de envío)
ts_duplicado = int(time.time() * 1000)
evento_duplicado = spark.createDataFrame(
    [("red.lectura", "nic-duplicado", 1234.5, 5678.9, 10.0, 25.0, 0.0, 0.0, 0.0, 0.0, 1000.0, ts_duplicado)],
    columnas,
)
salida = evento_duplicado.select(
    evento_duplicado.sensorId.cast("string").alias("key"),
    to_json(struct(*columnas)).alias("value"),
)
for intento in range(2):
    (salida.write.format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("topic", "red-telemetria")
        .save())
print(">>> 'nic-duplicado' publicado DOS veces con el mismo timestamp")

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+---------------+--------------+---+
|sensorId|bytesSentPerSec|uso_enlace_pct|ts |
+--------+---------------+--------------+---+
+--------+---------------+--------------+---+

-------------------------------------------
Batch: 1
-------------------------------------------
+----------------------------------+------------------+---------------------+-----------------------+
|sensorId                          |bytesSentPerSec   |uso_enlace_pct       |ts                     |
+----------------------------------+------------------+---------------------+-----------------------+
|Ethernet 2                        |0.0               |0.0                  |2026-10-08 18:20:41.388|
|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0                  |2026-10-08 18:20:40.375|
|Ethernet                          |3403.8321088494936|0.013432895670399313 |2026-10-08 18:20:41.388|
|Ethernet

In [44]:
# Momento 3: detener y revisar
import time
time.sleep(20)
consulta.stop()
print("Busca 'nic-duplicado' arriba: debe aparecer una sola vez, aunque se publicó dos veces.")

-------------------------------------------
Batch: 3
-------------------------------------------
+----------------------------------+------------------+---------------------+-----------------------+
|sensorId                          |bytesSentPerSec   |uso_enlace_pct       |ts                     |
+----------------------------------+------------------+---------------------+-----------------------+
|Ethernet 2                        |0.0               |0.0                  |2026-10-08 18:20:46.453|
|nic-duplicado                     |1234.5            |0.005530719999999999 |2026-10-08 18:20:46.3  |
|Ethernet 2                        |0.0               |0.0                  |2026-10-08 18:20:46.744|
|Ethernet                          |5102.975957049212 |0.07036702045265093  |2026-10-08 18:20:45.44 |
|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0                  |2026-10-08 18:20:45.44 |
|Ethernet                          |54.2693575968418  |9.551406937044155E-5 |2026-10-08

`dropDuplicates(["sensorId", "timestamp"])` descarta cualquier evento cuya combinación de NIC e instante ya se vio antes. Sin `withWatermark`, esta operación tendría que recordar **todos** los eventos vistos jamás — el mismo problema de 3.7, sobre los propios datos.

El watermark acota cuánto se recuerda: pasado el margen (30 segundos), un identificador viejo se olvida. Esto modela directamente la semántica de entrega: Kafka entrega `red-telemetria` **al menos una vez** (S7) — `dropDuplicates` con watermark convierte ese "al menos una vez" en un resultado que se comporta como **"exactamente una vez"**, sin que el bridge tenga que garantizar nada.

## 3.13 Intervalo de disparo: el costo de decidir cada cuánto procesar

**Producto del paso:** el mismo stream de tráfico de red, con dos ritmos de disparo distintos, y la diferencia real en cuántos micro-lotes se alcanzan a correr.

Esta comparación importa en un sistema de monitoreo de red: un disparo muy frecuente detecta picos de tráfico casi en tiempo real pero consume más CPU; uno lento agrupa más lecturas pero puede perder un pico corto.

In [45]:
def contar_microlotes(intervalo, segundos_totales):
    consulta = (
        eventos.select("sensorId", "bytesSentPerSec", "bytesRecvPerSec", "ts")
        .writeStream
        .outputMode("append")
        .format("memory")
        .queryName(f"conteo_{intervalo.replace(' ', '_')}")
        .trigger(processingTime=intervalo)
        .start()
    )
    consulta.awaitTermination(segundos_totales)
import time
time.sleep(20)
    consulta.stop()
    return consulta.lastProgress["batchId"] if consulta.lastProgress else None

ultimo_lote_rapido = contar_microlotes("1 second", 15)
ultimo_lote_lento  = contar_microlotes("6 seconds", 15)

print("Con disparo cada 1 segundo, último batchId en 15s:", ultimo_lote_rapido)
print("Con disparo cada 6 segundos, último batchId en 15s:", ultimo_lote_lento)

IndentationError: unexpected indent (346470716.py, line 14)

`format("memory")` guarda la salida en una tabla temporal en vez de imprimirla — útil para medir sin llenar la pantalla. `consulta.lastProgress["batchId"]` es el número del último micro-lote: con disparo de 1 segundo, en 15 segundos corren muchos más micro-lotes que con 6 segundos.

**Para el sistema de monitoreo de red (Proyecto Sello):**
- **Disparo rápido (1s):** ideal para detectar ráfagas de tráfico o picos de error en tiempo real, como un DDoS. Latencia baja, mayor overhead de procesamiento.
- **Disparo lento (6s):** mejor para reportes de tendencia de uso del enlace a lo largo del turno. Mayor throughput por lote, latencia más alta.

La elección depende del SLA: ¿el sistema necesita reaccionar en segundos (alertas) o resumir en minutos (reportes)?

**Error frecuente**: el log puede mostrar líneas en rojo `ERROR WriteToDataSourceV2Exec`, `TaskKilledException` o `Aborting task` — es ruido esperado del `consulta.stop()` dentro de `contar_microlotes()`, que corta la consulta a mitad de un micro-lote. Mientras la celda termine imprimiendo los dos `print()` finales, los resultados son correctos.

## 3.14 Documentar hallazgos y responder preguntas de reflexión

**Producto del paso:** una celda markdown que dice, en tus propias palabras, qué efecto tuvo cada parámetro sobre el monitoreo de tráfico de red.

Responde, con los resultados reales que obtuviste arriba:

1. En 3.7, ¿qué le pasaba a la tabla de tráfico impresa en cada micro-lote, y por qué dejó de pasar en 3.8?
2. En 3.11, ¿el evento `nic-tardio` (bytesSent=999999.9) apareció o no en la salida? ¿Cuántos segundos de diferencia había entre su `timestamp` y el watermark del momento en que se publicó?
3. En 3.12, ¿cuántas veces se publicó `nic-duplicado`, y cuántas veces apareció en la salida deduplicada?
4. En 3.13, ¿qué `batchId` alcanzó cada disparo (1 segundo vs. 6 segundos) en los mismos 15 segundos de reloj? ¿Cuál elegirías para alertas de tráfico en vivo, y cuál para un reporte de uso del enlace por hora?

**(Completa acá, con tus propios resultados)**

1.
2.
3.
4.